In [3]:
from datasets import load_dataset
dataset_twitter = load_dataset("Jacobvs/CelebrityTweets")

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset parquet downloaded and prepared to /Users/ivan/.cache/huggingface/datasets/Jacobvs___parquet/Jacobvs--CelebrityTweets-de2be27b448e7442/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
import pandas as pd
import ast
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [9]:
df = pd.DataFrame(dataset_twitter)


In [17]:
df = pd.read_csv("twitter.csv")
df.head()

,Unnamed: 0,train
0,0,"{'index': 0, 'date': '2022-12-01 19:24:25', 'i..."
1,1,"{'index': 1, 'date': '2022-09-07 01:14:34', 'i..."
2,2,"{'index': 2, 'date': '2022-07-07 19:33:51', 'i..."
3,3,"{'index': 3, 'date': '2022-02-16 03:05:25', 'i..."
4,4,"{'index': 4, 'date': '2021-12-30 22:33:21', 'i..."


In [16]:
df["train"] = df["train"].apply(ast.literal_eval)

tweets = pd.json_normalize(df["train"])

tweets.head()

,index,date,id,username,text
0,0,2022-12-01 19:24:25,1598397624750866432,justinbieber,long time coming and excited to finally announ...
1,1,2022-09-07 01:14:34,1567320386546589696,justinbieber,https://t.co/qRY7ltRkV0
2,2,2022-07-07 19:33:51,1545128982143651840,justinbieber,1 year of preparation 🔥 @FreeFire_NA #GarenaFr...
3,3,2022-02-16 03:05:25,1493783548276314115,justinbieber,"Go for the Gold, Ladies!!!!!! Cannot wait to w..."
4,4,2021-12-30 22:33:21,1476682851324239873,justinbieber,The countdown begins… https://t.co/S5qZP92Ziz


In [18]:
tweets = tweets[tweets["username"].isin(["elonmusk", "justinbieber"])]

tweets = tweets[["username", "text"]]
tweets.head()

,username,text
0,justinbieber,long time coming and excited to finally announ...
1,justinbieber,https://t.co/qRY7ltRkV0
2,justinbieber,1 year of preparation 🔥 @FreeFire_NA #GarenaFr...
3,justinbieber,"Go for the Gold, Ladies!!!!!! Cannot wait to w..."
4,justinbieber,The countdown begins… https://t.co/S5qZP92Ziz


In [19]:


def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

tweets["text"] = tweets["text"].apply(clean_text)

tweets = tweets[tweets["text"] != ""]

In [21]:
from sklearn.model_selection import train_test_split

X = tweets["text"]
y = tweets["username"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [22]:
vectorizer = TfidfVectorizer(ngram_range=(1,3))

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [23]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_vec, y_train)

pred_lr = lr.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr))

Accuracy: 0.8153846153846154
              precision    recall  f1-score   support

    elonmusk       0.75      0.94      0.83        32
justinbieber       0.92      0.70      0.79        33

    accuracy                           0.82        65
   macro avg       0.83      0.82      0.81        65
weighted avg       0.84      0.82      0.81        65



In [24]:
nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

pred_nb = nb.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, pred_nb))
print(classification_report(y_test, pred_nb))

Accuracy: 0.8
              precision    recall  f1-score   support

    elonmusk       0.83      0.75      0.79        32
justinbieber       0.78      0.85      0.81        33

    accuracy                           0.80        65
   macro avg       0.80      0.80      0.80        65
weighted avg       0.80      0.80      0.80        65



In [25]:
sentences = ["Can't wait for my new album to drop!","Starship is ready for launch!"]

vec = vectorizer.transform(sentences)

print("Logistic Regression:")
print(lr.predict(vec))

print("Naive Bayes:")
print(nb.predict(vec))

Logistic Regression:
['justinbieber' 'elonmusk']
Naive Bayes:
['justinbieber' 'elonmusk']


In [32]:
sentences = ["I just put dick into my ass","I think that im gay"]

vec = vectorizer.transform(sentences)

print("Logistic Regression:")
print(lr.predict(vec))

print("Naive Bayes:")
print(nb.predict(vec))

Logistic Regression:
['elonmusk' 'elonmusk']
Naive Bayes:
['elonmusk' 'elonmusk']


In [26]:
probs = lr.predict_proba(X_test_vec)
preds = lr.predict(X_test_vec)

classes = lr.classes_

for tweet, true_label, pred, prob in zip(X_test, y_test, preds, probs):

    print("Tweet:", tweet)
    print("True:", true_label)
    print("Pred:", pred)
    print(classes[0], prob[0])
    print(classes[1], prob[1])
    print()

Tweet: Ouch my feet!!
True: elonmusk
Pred: justinbieber
elonmusk 0.4193312012819701
justinbieber 0.5806687987180299

Tweet: My second collab with is dropping around the 🌎 tomorrow and it comes with socks
True: justinbieber
Pred: justinbieber
elonmusk 0.47624451787547917
justinbieber 0.5237554821245208

Tweet: On the set of
True: justinbieber
Pred: justinbieber
elonmusk 0.3616632924712879
justinbieber 0.6383367075287121

Tweet: Your album and doc are amazing . So happy and proud of you
True: justinbieber
Pred: justinbieber
elonmusk 0.4942365274328695
justinbieber 0.5057634725671305

Tweet: Watch me perform at The 23rd Annual A Home for the Holidays at this Sunday at 9:30PM ET/9:00PM PT on…
True: justinbieber
Pred: justinbieber
elonmusk 0.47818517078707345
justinbieber 0.5218148292129265

Tweet: Peaches video 9pm PST
True: justinbieber
Pred: justinbieber
elonmusk 0.3410515067916382
justinbieber 0.6589484932083618

Tweet: deamplify means shadowban
True: elonmusk
Pred: elonmusk
elonmusk 0.